<a href="https://colab.research.google.com/github/bvignesz/nifty-layer2-engine/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade fyers-apiv3 fastapi uvicorn requests nest_asyncio pandas scipy bottleneck

In [ ]:
!pip install -q --upgrade bottleneck

In [ ]:
# CELL 1: Install dependencies, cloudflared binary & mount Google Drive
!pip install -q fyers-apiv3 fastapi uvicorn requests nest_asyncio pandas scipy

import os
import urllib.request
from google.colab import drive

# 1. Install cloudflared binary directly for the mobile HTTPS tunnel
cf_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_path):
    print("📥 Downloading cloudflared binary...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf_path
    )
    os.chmod(cf_path, 0o755)
    print("✅ cloudflared installed!")

# 2. Mount Google Drive for permanent SQLite & CSV storage
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/Nifty50_Option_Engine"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Setup complete! All databases and CSVs will save to: {SAVE_DIR}")

In [ ]:
# CELL 2: Credentials & 1-Click FYERS Daily Token Generator
import os
from urllib.parse import urlparse, parse_qs
from fyers_apiv3 import fyersModel

# --- 1. YOUR CREDENTIALS ---
FYERS_CLIENT_ID = os.environ.get("FYERS_CLIENT_ID", "")
FYERS_SECRET_KEY = os.environ.get("FYERS_SECRET_KEY", "")
FYERS_REDIRECT_URI = "https://trade.fyers.in/api-login/redirect-uri/index.html"
FYERS_ACCESS_TOKEN = os.environ.get("FYERS_ACCESS_TOKEN", "")

# Active Nifty 50 Front-Month Futures Symbol on FYERS (Update monthly after last Tuesday expiry)
NIFTY_FUT_SYMBOL = "NSE:NIFTY26NOVFUT"

# Optional Telegram Bot Credentials (from @BotFather & @userinfobot)
TELEGRAM_BOT_TOKEN = os.environ.get("TELEGRAM_BOT_TOKEN", "")
TELEGRAM_CHAT_ID = os.environ.get("TELEGRAM_CHAT_ID", "")
RISK_PER_TRADE_RS = 5000.0

# --- 2. GENERATE TODAY'S ACCESS TOKEN IN COLAB ---
GENERATE_NEW_TOKEN = True  # Keep True to log in & auto-fill token

if GENERATE_NEW_TOKEN:
    session = fyersModel.SessionModel(
        client_id=FYERS_CLIENT_ID,
        secret_key=FYERS_SECRET_KEY,
        redirect_uri=FYERS_REDIRECT_URI,
        response_type="code",
        grant_type="authorization_code"
    )
    login_url = session.generate_authcode()
    print(f"👉 1. Tap this URL to log in to FYERS:\n{login_url}\n")
    raw_input_str = input("👉 2. Paste the FULL redirected URL (or just the 'auth_code') here: ").strip()

    # Automatically extract auth_code if you pasted the full browser URL
    if "auth_code=" in raw_input_str:
        auth_code_input = parse_qs(urlparse(raw_input_str).query)["auth_code"][0]
    else:
        auth_code_input = raw_input_str

    session.set_token(auth_code_input)
    token_resp = session.generate_token()
    if "access_token" in token_resp:
        FYERS_ACCESS_TOKEN = token_resp["access_token"]
        print("✅ Access Token generated and loaded successfully!")
    else:
        print("❌ Token error:", token_resp)

os.environ["FYERS_CLIENT_ID"] = FYERS_CLIENT_ID
os.environ["FYERS_ACCESS_TOKEN"] = FYERS_ACCESS_TOKEN
os.environ["NIFTY_FUT_SYMBOL"] = NIFTY_FUT_SYMBOL
os.environ["TELEGRAM_BOT_TOKEN"] = TELEGRAM_BOT_TOKEN
os.environ["TELEGRAM_CHAT_ID"] = TELEGRAM_CHAT_ID
os.environ["RISK_PER_TRADE_RS"] = str(RISK_PER_TRADE_RS)

In [ ]:
# CELL 3: Layer-2 Corrected Institutional 9-Check Engine (Nifty 50 & Sensex)
# FINAL OUTPUT STATES: Strictly 3 Distinct States -> [ 🟢 BULL | 🔴 BEAR | ⚪ NO TRADE ]
# INCLUDES: Auto-Futures Rollover (Nov->Dec->Jan) + Auto-Strike Scaling + Fixed Strike Variables
from collections import Counter
import csv
from datetime import datetime, time as dtime, timedelta
import json
import math
import os
import re
import socket
import sqlite3
import subprocess
import threading
import time
from typing import Any, Dict, List, Optional
from zoneinfo import ZoneInfo

from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse, JSONResponse
from fyers_apiv3 import fyersModel
import numpy as np
import pandas as pd
import requests
from scipy.stats import norm
import uvicorn


# ==============================================================================
# === STRATEGY 01: RESEARCH-ONLY TPO + VRZ + BOF; NO LIVE ORDER GATE ===
# ==============================================================================
IST = ZoneInfo("Asia/Kolkata")
SESSION_START = dtime(9, 15)
SESSION_END = dtime(15, 30)


def clean_bars(raw, now=None):
  """FYERS candles [epoch, open, high, low, close, volume] -> COMPLETED 5m IST bars."""
  now = now or datetime.now(IST)
  rows = []
  for bar in raw:
    if len(bar) < 6:
      continue
    ts = datetime.fromtimestamp(int(bar[0]), IST)
    if not (SESSION_START <= ts.time() < SESSION_END):
      continue
    if ts + timedelta(minutes=5, seconds=5) > now:
      continue
    o, h, l, c, v = (float(x) for x in bar[1:6])
    if (
        not all(math.isfinite(x) for x in (o, h, l, c, v))
        or l > h
        or not (l <= o <= h and l <= c <= h)
    ):
      continue
    rows.append(
        dict(ts=ts, open=o, high=h, low=l, close=c, volume=v, day=ts.date())
    )
  df = pd.DataFrame(rows)
  if df.empty:
    return df
  return (
      df.sort_values("ts")
      .drop_duplicates("ts", keep="last")
      .reset_index(drop=True)
  )


def bracket_id(ts):
  return ((ts.hour * 60 + ts.minute) - (9 * 60 + 15)) // 30


def tpo_profile(day_df, row_size=5.0):
  if day_df.empty or row_size <= 0:
    return None
  groups = day_df.groupby(day_df.ts.map(bracket_id))
  counts = Counter()
  for _, g in groups:
    low, high = float(g.low.min()), float(g.high.max())
    a = math.floor((low + 1e-9) / row_size)
    b = math.floor((high + 1e-9) / row_size)
    for index in range(a, b + 1):
      counts[index] += 1
  if not counts:
    return None
  mid = (day_df.low.min() + day_df.high.max()) / 2
  poc = min(counts, key=lambda i: (-counts[i], abs(i * row_size - mid), i))
  left = right = poc
  included = counts[poc]
  target = math.ceil(sum(counts.values()) * 0.70)
  while included < target:
    li, ri = left - 1, right + 1
    lv, rv = counts.get(li, 0), counts.get(ri, 0)
    if li < min(counts):
      lv = -1
    if ri > max(counts):
      rv = -1
    if lv < 0 and rv < 0:
      break
    if lv > rv or (
        lv == rv and abs(li * row_size - mid) <= abs(ri * row_size - mid)
    ):
      left = li
      included += max(lv, 0)
    else:
      right = ri
      included += max(rv, 0)
  return dict(
      poc=round(poc * row_size, 3),
      vah=round((right + 1) * row_size, 3),
      val=round(left * row_size, 3),
      total_tpos=sum(counts.values()),
      brackets=len(groups),
      rows=sorted((i * row_size, n) for i, n in counts.items()),
      convention="one-row outward, row interval [price,price+step)",
  )


def confirmed_vrz(df, half_width=5.0):
  result = []
  if df.empty:
    return result
  for day, g in df.groupby("day"):
    brackets = []
    for idx, x in g.groupby(g.ts.map(bracket_id)):
      if len(x) != 6:
        continue
      start = min(x.ts)
      end = max(x.ts) + timedelta(minutes=5)
      brackets.append(
          (idx, start, end, float(x.high.max()), float(x.low.min()))
      )
    brackets.sort()
    for i in range(2, len(brackets) - 2):
      chunk = brackets[i - 2 : i + 3]
      if any(chunk[j + 1][0] - chunk[j][0] != 1 for j in range(4)):
        continue
      cur = chunk[2]
      if all(cur[3] > x[3] for j, x in enumerate(chunk) if j != 2):
        kind, px = "HIGH", cur[3]
      elif all(cur[4] < x[4] for j, x in enumerate(chunk) if j != 2):
        kind, px = "LOW", cur[4]
      else:
        continue
      result.append(
          dict(
              kind=kind,
              lower=px - half_width,
              upper=px + half_width,
              center=px,
              formed_at=cur[2].isoformat(),
              available_at=chunk[4][2].isoformat(),
              day=str(day),
          )
      )
  return result


def classify_bof(today, zones, now=None):
  events = []
  if today.empty:
    return events
  seen = set()
  for i in range(1, len(today)):
    prev, cur = today.iloc[i - 1], today.iloc[i]
    if cur.ts - prev.ts != timedelta(minutes=5):
      continue
    for z in zones:
      if z["id"] in seen:
        continue
      if datetime.fromisoformat(z["available_at"]) > prev.ts:
        continue
      lo, hi = z["lower"], z["upper"]
      side = None
      if (
          prev.close > hi
          and cur.close <= hi
          and cur.close >= lo
          and i >= 2
          and today.iloc[i - 2].close <= hi
      ):
        side = "PE"
      if (
          prev.close < lo
          and cur.close >= lo
          and cur.close <= hi
          and i >= 2
          and today.iloc[i - 2].close >= lo
      ):
        side = "CE"
      if not side:
        continue
      strict = cur.close < prev.low if side == "PE" else cur.close > prev.high
      events.append(
          dict(
              zone=z["id"],
              side=side,
              breakout_time=prev.ts.isoformat(),
              reclaim_time=cur.ts.isoformat(),
              strict=bool(strict),
              stop=float(
                  max(prev.high, cur.high)
                  if side == "PE"
                  else min(prev.low, cur.low)
              ),
              status="RESEARCH_EVENT",
          )
      )
      seen.add(z["id"])
  return events


def evaluate(raw, now=None, row_size=5.0):
  now = now or datetime.now(IST)
  df = clean_bars(raw, now)
  base = dict(
      status="INSUFFICIENT_DATA",
      decision="NO_TRADE",
      model="SHADOW_ONLY",
      reason="",
      previous_tpo=None,
      vrz_count=0,
      events=[],
      last_bar=None,
  )
  if df.empty:
    base["reason"] = "No completed valid 5-minute bars"
    return base
  base["last_bar"] = df.iloc[-1].ts.isoformat()
  if now - (df.iloc[-1].ts + timedelta(minutes=5)) > timedelta(minutes=15):
    base["reason"] = "Stale 5-minute price bars"
    return base
  dates = sorted(set(df.day))
  today = now.date()
  completed = [d for d in dates if d < today]
  if today not in dates or not completed:
    base["reason"] = "Current and previous completed sessions required"
    return base
  prev_date = completed[-1]
  prev = df[df.day == prev_date]
  expected = [
      datetime.combine(prev_date, SESSION_START, tzinfo=IST)
      + timedelta(minutes=5 * k)
      for k in range(75)
  ]
  if len(prev) != 75 or list(prev.ts) != expected:
    base["reason"] = (
        "Previous session not complete at configured session hours"
    )
    return base
  profile = tpo_profile(prev, row_size)
  base["previous_tpo"] = {k: v for k, v in profile.items() if k != "rows"}
  cur = df[df.day == today]
  zones = [
      dict(
          id="PDVAH",
          lower=profile["vah"] - row_size,
          upper=profile["vah"],
          available_at=datetime.combine(
              today, SESSION_START, tzinfo=IST
          ).isoformat(),
      ),
      dict(
          id="PDVAL",
          lower=profile["val"],
          upper=profile["val"] + row_size,
          available_at=datetime.combine(
              today, SESSION_START, tzinfo=IST
          ).isoformat(),
      ),
  ]
  vrzs = confirmed_vrz(df[df.day <= today], half_width=row_size)
  for k, z in enumerate(vrzs):
    if datetime.fromisoformat(z["available_at"]) <= now and z["kind"] in (
        "HIGH",
        "LOW",
    ):
      zones.append(
          dict(
              id=f'VRZ_{k}_{z["kind"]}',
              lower=z["lower"],
              upper=z["upper"],
              available_at=z["available_at"],
          )
      )
  base["vrz_count"] = len(zones) - 2
  base["events"] = classify_bof(cur, zones)
  base["status"] = "OK"
  base["decision"] = "RESEARCH_ONLY"
  base["reason"] = "Historical events shown; not a trade approval"
  return base


# ==============================================================================
# 1. CONFIGURABLE HYPOTHESIS THRESHOLDS & INDEX SPECIFICATIONS
# ==============================================================================
SAVE_DIR = "/content/drive/MyDrive/Nifty50_Option_Engine"
os.makedirs(SAVE_DIR, exist_ok=True)

DB_PATH = os.path.join(SAVE_DIR, "nifty_oc_history.db")
LIVE_CSV_PATH = os.path.join(SAVE_DIR, "nifty_live_chain.csv")
SCORECARD_CSV_PATH = os.path.join(SAVE_DIR, "nifty_6check_scorecard.csv")
SIGNALS_CSV_PATH = os.path.join(SAVE_DIR, "nifty_signals_log.csv")
AUDIT_CSV_PATH = os.path.join(SAVE_DIR, "nifty_layer2_corrected_audit.csv")

ACTIVE_INDEX = os.environ.get("ACTIVE_INDEX", "NIFTY").upper()
if ACTIVE_INDEX == "SENSEX":
  NIFTY_SPOT_SYM = "BSE:SENSEX-INDEX"
  NIFTY_LOT_SIZE = int(os.environ.get("SENSEX_LOT_SIZE", "20"))
  NIFTY_STRIKE_STEP = 100
  MAX_ALLOWED_SPREAD = 2.50
  DEFAULT_STOP_PTS = 110.0
else:
  NIFTY_SPOT_SYM = "NSE:NIFTY50-INDEX"
  NIFTY_LOT_SIZE = int(os.environ.get("NIFTY_LOT_SIZE", "75"))
  NIFTY_STRIKE_STEP = 50
  MAX_ALLOWED_SPREAD = 0.75
  DEFAULT_STOP_PTS = 35.0

TOP5_WEIGHTS: Dict[str, float] = {
    "NSE:HDFCBANK-EQ": 0.118,
    "NSE:RELIANCE-EQ": 0.092,
    "NSE:ICICIBANK-EQ": 0.081,
    "NSE:BHARTIARTL-EQ": 0.041,
    "NSE:LT-EQ": 0.039,
}
TOP5_STOCKS = list(TOP5_WEIGHTS.keys())

HYPOTHESIS_THRESHOLDS: Dict[str, float] = {
    "wall_gap_ratio_bull": float(os.environ.get("HYP_WALL_RATIO_BULL", "1.20")),
    "wall_gap_ratio_bear": float(os.environ.get("HYP_WALL_RATIO_BEAR", "1.20")),
    "opposing_cut_pct_3m": float(
        os.environ.get("HYP_OPPOSING_CUT_3M", "-1.20")
    ),
    "supporting_oi_speed": float(os.environ.get("HYP_OI_SPEED_MULT", "1.40")),
    "max_churn_ratio": float(os.environ.get("HYP_MAX_CHURN", "12.0")),
    "min_doi_lots_for_churn": float(
        os.environ.get("HYP_MIN_DOI_LOTS", "25.0")
    ),
    "min_weighted_breadth": float(
        os.environ.get("HYP_WEIGHTED_BREADTH", "0.55")
    ),
    "atm_flip_vol_mult": float(os.environ.get("HYP_FLIP_VOL_MULT", "1.20")),
    "max_snapshot_staleness_sec": float(
        os.environ.get("HYP_MAX_STALE_SEC", "300.0")
    ),
}

MIN_DTE_INTRADAY = int(os.environ.get("MIN_DTE_INTRADAY", "1"))

LATEST_STATE: Dict[str, Any] = {
    "status": "INITIALIZING",
    "error": None,
    "final_verdict": "NO TRADE",  # Strictly: "BULL" | "BEAR" | "NO TRADE"
    "banner": "INITIALIZING...",
    "decision_outcome": "NO TRADE",
    "scorecard": [],
    "meta": {},
    "layer2_audit": {},
    "trade_plan": None,
}


def now_ist() -> datetime:
  return datetime.now(IST)


def get_fyers_client() -> fyersModel.FyersModel:
  return fyersModel.FyersModel(
      client_id=os.environ["FYERS_CLIENT_ID"].strip(),
      is_async=False,
      token=os.environ["FYERS_ACCESS_TOKEN"].strip(),
      log_path="",
  )


def get_active_front_month_fut(
    fyers: fyersModel.FyersModel, now_dt: datetime, index_mode: str = "NIFTY"
) -> str:
  """Automatically resolves active front-month futures ticker (e.g., 26NOVFUT -> 26DECFUT -> 27JANFUT)."""
  env_sym = os.environ.get("NIFTY_FUT_SYMBOL", "").strip()
  months = [
      "JAN",
      "FEB",
      "MAR",
      "APR",
      "MAY",
      "JUN",
      "JUL",
      "AUG",
      "SEP",
      "OCT",
      "NOV",
      "DEC",
  ]
  cur_yy = now_dt.year % 100
  cur_m_idx = now_dt.month - 1
  next_yy = (now_dt.year + 1) % 100 if cur_m_idx == 11 else cur_yy
  next_m_idx = (cur_m_idx + 1) % 12

  prefix = "BSE:SENSEX" if index_mode == "SENSEX" else "NSE:NIFTY"
  cur_cand = f"{prefix}{cur_yy:02d}{months[cur_m_idx]}FUT"
  next_cand = f"{prefix}{next_yy:02d}{months[next_m_idx]}FUT"

  candidates = [c for c in [cur_cand, next_cand, env_sym] if c]
  try:
    q = fyers.quotes(data={"symbols": ",".join(dict.fromkeys(candidates))})
    valid_contracts = []
    for item in q.get("d", []):
      v = item.get("v", {})
      if item.get("s") == "ok" and float(v.get("lp", 0) or 0) > 0:
        valid_contracts.append(item.get("n"))
    if cur_cand in valid_contracts:
      return cur_cand
    if next_cand in valid_contracts:
      return next_cand
    if valid_contracts:
      return valid_contracts[0]
  except Exception:
    pass
  return env_sym or cur_cand


# ==============================================================================
# 2. INITIALIZE SQLITE DB & CSV HEADERS
# ==============================================================================
def init_storage():
  with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
            CREATE TABLE IF NOT EXISTS header_snapshots (
                ts INTEGER PRIMARY KEY, ist_time TEXT, spot REAL, vix REAL, vix_chp REAL,
                fut_close REAL, fut_vwap REAL, fut_oi INTEGER,
                top5_bull INTEGER, top5_bear INTEGER, expiry_date TEXT, dte INTEGER
            )
        """)
    conn.execute("""
            CREATE TABLE IF NOT EXISTS strike_snapshots (
                ts INTEGER, strike INTEGER, opt_type TEXT, symbol TEXT,
                ltp REAL, bid REAL, ask REAL, oi INTEGER, oich INTEGER, oichp REAL,
                volume INTEGER, iv REAL, delta REAL, gamma REAL, theta REAL, vega REAL,
                PRIMARY KEY (ts, strike, opt_type)
            )
        """)
    conn.commit()

  expected_header = [
      "IST_Time",
      "Data_Mode",
      "Signal_Banner",
      "Bull_Score",
      "Bear_Score",
      "Bull_Checks_Passed",
      "Bear_Checks_Passed",
      "Spot",
      "Fut_Close",
      "Fut_VWAP",
      "Fut_Quad",
      "VIX",
      "VIX_Chg_Pct",
      "Net_GEX_Cr",
      "GEX_Regime",
      "Put_Wall",
      "Call_Wall",
      "ATM_Wall_Status",
      "ATM_Seller_Lead",
      "ATM_3m_Burst",
      "Top5_Bull",
      "Top5_Bear",
      "Rec_Contract",
      "Entry_Ask",
      "Option_SL",
      "Lots",
      "Est_Risk_Rs",
  ]
  if not os.path.exists(SIGNALS_CSV_PATH):
    with open(SIGNALS_CSV_PATH, "w", newline="") as f:
      csv.writer(f).writerow(expected_header)

  audit_header = [
      "IST_Time",
      "Final_Verdict",
      "Signal_Banner",
      "Active_Candidate_Side",
      "Bull_Raw_9",
      "Bear_Raw_9",
      "Bull_Independent_Pillars_4",
      "Bear_Independent_Pillars_4",
      "No_Trade_Reasons",
      "Freshness_OK",
      "Spot",
      "Fut_Close",
      "Fut_VWAP",
      "Weighted_Bull_Breadth",
      "Weighted_Bear_Breadth",
      "Wall_Migration_Type",
      "ATM_Candidate_Label",
      "GEX_Est_Cr_Per1Pct",
      "VEX_Est_Cr_Per1PctIV",
      "CEX_Est_Cr_PerDay",
      "Gamma_Flip_Scenario",
      "Rec_Contract",
  ]
  if not os.path.exists(AUDIT_CSV_PATH):
    with open(AUDIT_CSV_PATH, "w", newline="") as f:
      csv.writer(f).writerow(audit_header)


# ==============================================================================
# 3. CONSISTENT GREEKS & EXPLICITLY SCALED SCENARIO EXPOSURES (CHECK #8)
# ==============================================================================
def get_greeks_safe(
    leg: Dict[str, Any],
    spot: float,
    dte: int,
    vix_val: float,
    fut_price: Optional[float] = None,
) -> Dict[str, float]:
  g = leg.get("greeks") if isinstance(leg.get("greeks"), dict) else {}
  delta = float(g.get("delta", 0.0) or 0.0)
  gamma = float(g.get("gamma", 0.0) or 0.0)
  theta = float(g.get("theta", 0.0) or 0.0)
  vega = float(g.get("vega", 0.0) or 0.0)
  iv = float(g.get("iv", 0.0) or 0.0)

  F = max(float(fut_price or spot or 1.0), 1.0)
  K = max(float(leg.get("strike_price", F) or F), 1.0)
  T = max(float(dte) / 365.0, 1.0 / (365.0 * 4.0))
  sigma = max((iv if iv > 0 else vix_val) / 100.0, 0.08)
  r = 0.068
  sqrt_T = math.sqrt(T)
  disc = math.exp(-r * T)

  d1 = (math.log(F / K) + 0.5 * (sigma**2) * T) / (sigma * sqrt_T)
  d2 = d1 - sigma * sqrt_T
  pdf_d1 = norm.pdf(d1)

  if gamma == 0.0 or delta == 0.0:
    if leg.get("option_type") == "CE":
      delta = disc * float(norm.cdf(d1))
      theta = (
          -(F * disc * pdf_d1 * sigma) / (2 * sqrt_T)
          - r * K * disc * float(norm.cdf(d2))
      ) / 365.0
    else:
      delta = -disc * float(norm.cdf(-d1))
      theta = (
          -(F * disc * pdf_d1 * sigma) / (2 * sqrt_T)
          + r * K * disc * float(norm.cdf(-d2))
      ) / 365.0
    gamma = (disc * pdf_d1) / (F * sigma * sqrt_T)
    vega = F * disc * pdf_d1 * sqrt_T * 0.01

  vanna = (-disc * pdf_d1 * (d2 / sigma)) * 0.01
  charm_term = (
      pdf_d1 * (2.0 * r * T - d2 * sigma * sqrt_T) / (2.0 * T * sigma * sqrt_T)
  )
  if leg.get("option_type") == "CE":
    charm = -charm_term / 365.0
  else:
    charm = (-charm_term - r * disc * float(norm.cdf(-d2))) / 365.0

  return {
      "iv": round(sigma * 100, 2),
      "delta": round(delta, 4),
      "gamma": round(gamma, 6),
      "theta": round(theta, 2),
      "vega": round(vega, 2),
      "vanna": round(vanna, 5),
      "charm": round(charm, 5),
  }


def compute_swept_gamma_flip_scenario(
    strikes: List[int],
    ce_map: Dict[int, Any],
    pe_map: Dict[int, Any],
    fut_price: float,
    dte: int,
    vix_val: float,
    lot_size: int,
) -> float:
  if not strikes or fut_price <= 0:
    return fut_price
  T = max(float(dte) / 365.0, 1.0 / (365.0 * 4.0))
  sqrt_T = math.sqrt(T)
  r = 0.068
  disc = math.exp(-r * T)

  def net_gex_at(f_hyp: float) -> float:
    f_hyp = max(f_hyp, 1.0)
    total = 0.0
    for k in strikes:
      if k <= 0:
        continue
      c_leg, p_leg = ce_map.get(k, {}), pe_map.get(k, {})
      c_oi_units = int(c_leg.get("oi", 0) or 0)
      p_oi_units = int(p_leg.get("oi", 0) or 0)
      c_iv = max(
          float(c_leg.get("greeks", {}).get("iv", vix_val) or vix_val) / 100.0,
          0.08,
      )
      p_iv = max(
          float(p_leg.get("greeks", {}).get("iv", vix_val) or vix_val) / 100.0,
          0.08,
      )
      d1_c = (math.log(f_hyp / k) + 0.5 * (c_iv**2) * T) / (c_iv * sqrt_T)
      d1_p = (math.log(f_hyp / k) + 0.5 * (p_iv**2) * T) / (p_iv * sqrt_T)
      gam_c = (disc * norm.pdf(d1_c)) / (f_hyp * c_iv * sqrt_T)
      gam_p = (disc * norm.pdf(d1_p)) / (f_hyp * p_iv * sqrt_T)
      total += 0.01 * (f_hyp**2) * (gam_c * c_oi_units - gam_p * p_oi_units) / 1e7
    return total

  grid = [fut_price * (1.0 + 0.004 * i) for i in range(-20, 21)]
  vals = [net_gex_at(x) for x in grid]
  best_flip = grid[min(range(len(grid)), key=lambda idx: abs(vals[idx]))]
  for i in range(len(grid) - 1):
    if (vals[i] <= 0 <= vals[i + 1]) or (vals[i] >= 0 >= vals[i + 1]):
      denom = vals[i + 1] - vals[i]
      if abs(denom) > 1e-9:
        best_flip = grid[i] - vals[i] * (grid[i + 1] - grid[i]) / denom
      break
  return round(best_flip, 1)


def to_contract_lots(raw_qty: float, lot_size: int) -> float:
  return float(raw_qty) / max(float(lot_size), 1.0)


# ==============================================================================
# 4. LAYER-2 CORRECTED 9-CHECK ENGINE -> STRICTLY [ BULL | BEAR | NO TRADE ]
# ==============================================================================
def poll_and_evaluate(
    l2_trigger_side: Optional[str] = None,
    custom_stop_spot: Optional[float] = None,
    custom_target_spot: Optional[float] = None,
) -> Dict[str, Any]:
  global LATEST_STATE
  fyers = get_fyers_client()
  now_ts = int(time.time())
  now_dt = now_ist()
  ist_str = now_dt.strftime("%Y-%m-%d %H:%M:%S IST")
  th = HYPOTHESIS_THRESHOLDS

  # A. Fetch Expiry List & Validate Contract Expiry
  init_oc = fyers.optionchain(
      data={"symbol": NIFTY_SPOT_SYM, "strikecount": 1, "timestamp": ""}
  )
  if init_oc.get("s") != "ok" or not init_oc.get("data", {}).get("expiryData"):
    err_msg = f"NO TRADE: FYERS Expiry API Error at {ist_str}: {init_oc}"
    LATEST_STATE["error"] = err_msg
    LATEST_STATE["final_verdict"] = "NO TRADE"
    LATEST_STATE["decision_outcome"] = "NO TRADE"
    return LATEST_STATE

  expiries = init_oc["data"]["expiryData"]
  today_ist = now_dt.date()
  chosen_exp, chosen_dte = expiries[0], 1
  for exp in expiries:
    exp_dt = datetime.strptime(exp["date"], "%d-%m-%Y").date()
    dte = (exp_dt - today_ist).days
    if dte >= MIN_DTE_INTRADAY:
      chosen_exp, chosen_dte = exp, max(dte, 1)
      break

  # B. Fetch 10 Strikes Around Live ATM with Greeks (Auto-scales as Nifty moves!)
  oc_resp = fyers.optionchain(
      data={
          "symbol": NIFTY_SPOT_SYM,
          "strikecount": 10,
          "timestamp": str(chosen_exp["expiry"]),
          "greeks": "1",
      }
  )
  data = oc_resp.get("data", {})
  chain = data.get("optionsChain", [])
  if not chain:
    err_msg = f"NO TRADE: Empty optionsChain at {ist_str}"
    LATEST_STATE["error"] = err_msg
    LATEST_STATE["final_verdict"] = "NO TRADE"
    LATEST_STATE["decision_outcome"] = "NO TRADE"
    return LATEST_STATE

  spot = next(
      (
          float(c["ltp"])
          for c in chain
          if c.get("option_type") == "" or int(c.get("strike_price", 0)) == -1
      ),
      0.0,
  )
  vix_data = data.get("indiavixData", {})
  vix_ltp = float(vix_data.get("ltp", 0.0) or 0.0)
  vix_chp = float(vix_data.get("ltpchp", 0.0) or 0.0)

  # C. Weighted Top-5 Constituent Breadth with Synchronized Quotes
  bull_cnt, bear_cnt = 0, 0
  bull_weight_sum, bear_weight_sum, total_weight_seen = 0.0, 0.0, 0.0
  quotes_fresh = False
  try:
    q_resp = fyers.quotes(data={"symbols": ",".join(TOP5_STOCKS)})
    for item in q_resp.get("d", []):
      sym_name = item.get("n", "")
      v = item.get("v", {})
      lp = float(v.get("lp", 0.0) or 0.0)
      vwap = float(v.get("atp", 0.0) or 0.0)
      w = TOP5_WEIGHTS.get(sym_name, 0.05)
      if lp > 0 and vwap > 0:
        total_weight_seen += w
        if lp > vwap:
          bull_cnt += 1
          bull_weight_sum += w
        elif lp < vwap:
          bear_cnt += 1
          bear_weight_sum += w
    quotes_fresh = total_weight_seen > 0.15
  except Exception:
    quotes_fresh = False

  norm_bull_breadth = round(
      bull_weight_sum / max(total_weight_seen, 1e-6), 2
  )
  norm_bear_breadth = round(
      bear_weight_sum / max(total_weight_seen, 1e-6), 2
  )

  # D. Automatic Monthly Futures Rollover (Nov -> Dec -> Jan) & Freshness Check
  fut_sym = get_active_front_month_fut(fyers, now_dt, ACTIVE_INDEX)
  today_s = today_ist.strftime("%Y-%m-%d")
  from_s = (today_ist - timedelta(days=4)).strftime("%Y-%m-%d")
  futures_fresh = False
  fut_close, fut_vwap, fut_oi = 0.0, 0.0, 0
  fut_price_chg_3m, fut_oi_chg_3m = 0.0, 0
  open_gap_bull, open_gap_bear = False, False

  try:
    h_resp = fyers.history(
        data={
            "symbol": fut_sym,
            "resolution": "3",
            "date_format": "1",
            "range_from": from_s,
            "range_to": today_s,
            "cont_flag": "1",
            "oi_flag": "1",
        }
    )
    candles = h_resp.get("candles", [])
    if candles and len(candles) >= 3:
      last_bar_ts = int(candles[-1][0])
      bar_age_sec = now_ts - last_bar_ts
      futures_fresh = bar_age_sec <= (th["max_snapshot_staleness_sec"] + 180)
      today_epoch = int(
          datetime(
              today_ist.year, today_ist.month, today_ist.day, 9, 15, tzinfo=IST
          ).timestamp()
      )
      day_bars = [c for c in candles if c[0] >= today_epoch] or candles[-25:]
      prev_bars = [c for c in candles if c[0] < today_epoch]
      prev_close = (
          float(prev_bars[-1][4]) if prev_bars else float(day_bars[0][1])
      )
      day_open = float(day_bars[0][1])

      cum_pv = sum(
          ((c[2] + c[3] + c[4]) / 3.0) * max(c[5], 1) for c in day_bars
      )
      cum_vol = sum(max(c[5], 1) for c in day_bars)
      fut_vwap = round(cum_pv / max(cum_vol, 1.0), 2)
      fut_close = float(day_bars[-1][4])
      fut_oi = int(day_bars[-1][6]) if len(day_bars[-1]) > 6 else 0

      prev_3m_bar = day_bars[max(0, len(day_bars) - 2)]
      fut_price_chg_3m = round(fut_close - float(prev_3m_bar[4]), 2)
      fut_oi_chg_3m = (
          (fut_oi - int(prev_3m_bar[6])) if len(prev_3m_bar) > 6 else 0
      )

      if day_open >= prev_close:
        open_gap_bull = (fut_close >= day_open) or (fut_close > fut_vwap)
        open_gap_bear = (fut_close < prev_close) and (fut_close < fut_vwap)
      else:
        open_gap_bear = (fut_close <= day_open) or (fut_close < fut_vwap)
        open_gap_bull = (fut_close > prev_close) and (fut_close > fut_vwap)
  except Exception:
    futures_fresh = False

  # E. Store Current Snapshot & Fetch Synchronized t-1 (3m) and t-2 (6m) Snapshots
  strikes = sorted({
      int(c["strike_price"])
      for c in chain
      if c.get("option_type") in ("CE", "PE") and int(c["strike_price"]) > 0
  })
  atm = min(strikes, key=lambda s: abs(s - spot))
  ce_map = {
      int(c["strike_price"]): c for c in chain if c.get("option_type") == "CE"
  }
  pe_map = {
      int(c["strike_price"]): c for c in chain if c.get("option_type") == "PE"
  }

  eff_fut_for_greeks = fut_close if fut_close > 0 else spot

  with sqlite3.connect(DB_PATH) as conn:
    conn.execute(
        "INSERT OR REPLACE INTO header_snapshots VALUES"
        " (?,?,?,?,?,?,?,?,?,?,?,?)",
        (
            now_ts,
            ist_str,
            spot,
            vix_ltp,
            vix_chp,
            eff_fut_for_greeks,
            fut_vwap if fut_vwap > 0 else spot,
            fut_oi,
            bull_cnt,
            bear_cnt,
            chosen_exp["date"],
            chosen_dte,
        ),
    )
    for leg in chain:
      otype = leg.get("option_type", "")
      if otype not in ("CE", "PE"):
        continue
      s_val = int(leg["strike_price"])
      g = get_greeks_safe(
          leg, spot, chosen_dte, max(vix_ltp, 12.0), fut_price=eff_fut_for_greeks
      )
      leg["greeks"] = g
      conn.execute(
          "INSERT OR REPLACE INTO strike_snapshots VALUES"
          " (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
          (
              now_ts,
              s_val,
              otype,
              leg.get("symbol", ""),
              float(leg.get("ltp", 0)),
              float(leg.get("bid", 0)),
              float(leg.get("ask", 0)),
              int(leg.get("oi", 0)),
              int(leg.get("oich", 0) or 0),
              float(leg.get("oichp", 0.0) or 0.0),
              int(leg.get("volume", 0)),
              g["iv"],
              g["delta"],
              g["gamma"],
              g["theta"],
              g["vega"],
          ),
      )
    conn.commit()

    recent_headers = conn.execute(
        "SELECT ts, spot, fut_close, fut_vwap, fut_oi, expiry_date FROM"
        " header_snapshots WHERE ts < ? AND expiry_date = ? ORDER BY ts DESC"
        " LIMIT 20",
        (now_ts, chosen_exp["date"]),
    ).fetchall()

    snap_t1, snap_t2 = {}, {}
    header_t1, header_t2 = None, None
    if len(recent_headers) >= 1:
      header_t1 = recent_headers[0]
      for r in conn.execute(
          "SELECT strike, opt_type, oi, volume, ltp, iv FROM strike_snapshots"
          " WHERE ts = ?",
          (header_t1[0],),
      ):
        snap_t1[(int(r[0]), str(r[1]))] = {
            "oi": int(r[2]),
            "vol": int(r[3]),
            "ltp": float(r[4]),
            "iv": float(r[5]),
        }
    if len(recent_headers) >= 2:
      header_t2 = recent_headers[1]
      for r in conn.execute(
          "SELECT strike, opt_type, oi, volume, ltp, iv FROM strike_snapshots"
          " WHERE ts = ?",
          (header_t2[0],),
      ):
        snap_t2[(int(r[0]), str(r[1]))] = {
            "oi": int(r[2]),
            "vol": int(r[3]),
            "ltp": float(r[4]),
            "iv": float(r[5]),
        }

    cur_bracket = bracket_id(now_dt)
    hist_pos_pe_doi, hist_pos_ce_doi = [], []
    for idx_h in range(len(recent_headers) - 1):
      ts_a, ts_b = recent_headers[idx_h][0], recent_headers[idx_h + 1][0]
      if ts_a - ts_b > 360:
        continue
      dt_a = datetime.fromtimestamp(ts_a, IST)
      if abs(bracket_id(dt_a) - cur_bracket) <= 1:
        rows_a = {
            (int(r[0]), str(r[1])): int(r[2])
            for r in conn.execute(
                "SELECT strike, opt_type, oi FROM strike_snapshots WHERE ts=?",
                (ts_a,),
            )
        }
        rows_b = {
            (int(r[0]), str(r[1])): int(r[2])
            for r in conn.execute(
                "SELECT strike, opt_type, oi FROM strike_snapshots WHERE ts=?",
                (ts_b,),
            )
        }
        for (k_s, ot_s), oi_a in rows_a.items():
          if (k_s, ot_s) in rows_b and abs(k_s - atm) <= 2 * NIFTY_STRIKE_STEP:
            diff_v = oi_a - rows_b[(k_s, ot_s)]
            if ot_s == "PE" and diff_v > 0:
              hist_pos_pe_doi.append(diff_v)
            elif ot_s == "CE" and diff_v > 0:
              hist_pos_ce_doi.append(diff_v)

  has_2bar_sync = (
      header_t1 is not None
      and header_t2 is not None
      and (now_ts - int(header_t1[0]) <= th["max_snapshot_staleness_sec"])
      and (int(header_t1[0]) - int(header_t2[0]) <= th["max_snapshot_staleness_sec"])
  )
  data_mode = (
      "SYNC_2BAR_3M"
      if has_2bar_sync
      else ("SYNC_1BAR_3M" if header_t1 is not None else "WARMUP_WAIT_2BARS")
  )

  if not futures_fresh and header_t1 is not None and float(header_t1[2]) > 0:
    fut_close = float(header_t1[2])
    fut_vwap = float(header_t1[3])
    fut_price_chg_3m = round(spot - float(header_t1[1]), 2)
    fut_oi_chg_3m = fut_oi - int(header_t1[4])
    futures_fresh = True

  # F. Consistent Greek Exposures with Explicit Units & Scenario Labels
  net_gex_cr, net_vex_cr, net_cex_cr = 0.0, 0.0, 0.0
  call_gex_map, put_gex_map, live_rows = {}, {}, []
  for s in strikes:
    c_leg, p_leg = ce_map.get(s, {}), pe_map.get(s, {})
    c_oi_u, p_oi_u = int(c_leg.get("oi", 0) or 0), int(p_leg.get("oi", 0) or 0)
    c_g, p_g = c_leg.get("greeks", {}), p_leg.get("greeks", {})

    cg = 0.01 * (eff_fut_for_greeks**2) * float(c_g.get("gamma", 0.0)) * c_oi_u / 1e7
    pg = 0.01 * (eff_fut_for_greeks**2) * float(p_g.get("gamma", 0.0)) * p_oi_u / 1e7
    call_gex_map[s], put_gex_map[s] = cg, pg
    net_gex_cr += cg - pg

    net_vex_cr += (
        (
            float(c_g.get("vanna", 0.0)) * c_oi_u
            - float(p_g.get("vanna", 0.0)) * p_oi_u
        )
        * eff_fut_for_greeks
        / 1e7
    )
    net_cex_cr += (
        (
            float(c_g.get("charm", 0.0)) * c_oi_u
            - float(p_g.get("charm", 0.0)) * p_oi_u
        )
        * eff_fut_for_greeks
        / 1e7
    )

    if s in ce_map and s in pe_map:
      live_rows.append({
          "IST_Time": ist_str,
          "Expiry": chosen_exp["date"],
          "Spot": spot,
          "Strike": s,
          "CE_LTP": c_leg.get("ltp", 0),
          "CE_Bid": c_leg.get("bid", 0),
          "CE_Ask": c_leg.get("ask", 0),
          "CE_OI": c_oi_u,
          "CE_OICH": c_leg.get("oich", 0),
          "CE_OICHP": c_leg.get("oichp", 0),
          "CE_IV": c_g.get("iv", 0),
          "CE_Delta": c_g.get("delta", 0),
          "CE_Gamma": c_g.get("gamma", 0),
          "PE_LTP": p_leg.get("ltp", 0),
          "PE_Bid": p_leg.get("bid", 0),
          "PE_Ask": p_leg.get("ask", 0),
          "PE_OI": p_oi_u,
          "PE_OICH": p_leg.get("oich", 0),
          "PE_OICHP": p_leg.get("oichp", 0),
          "PE_IV": p_g.get("iv", 0),
          "PE_Delta": p_g.get("delta", 0),
          "PE_Gamma": p_g.get("gamma", 0),
          "Strike_Net_GEX_Cr": round(cg - pg, 2),
      })
  pd.DataFrame(live_rows).to_csv(LIVE_CSV_PATH, index=False)

  up_strikes = [s for s in strikes if s >= atm] or [atm]
  dn_strikes = [s for s in strikes if s <= atm] or [atm]
  call_wall = max(up_strikes, key=lambda s: int(ce_map.get(s, {}).get("oi", 0)))
  put_wall = max(dn_strikes, key=lambda s: int(pe_map.get(s, {}).get("oi", 0)))
  gamma_flip = compute_swept_gamma_flip_scenario(
      strikes,
      ce_map,
      pe_map,
      eff_fut_for_greeks,
      chosen_dte,
      max(vix_ltp, 12.0),
      NIFTY_LOT_SIZE,
  )
  gex_regime = (
      "POS_GEX_SCENARIO_EST"
      if (net_gex_cr >= 0 and eff_fut_for_greeks >= gamma_flip)
      else "NEG_GEX_SCENARIO_EST"
  )

  straddle_cost = float(ce_map.get(atm, {}).get("ltp", 0)) + float(
      pe_map.get(atm, {}).get("ltp", 0)
  )
  implied_daily_range = round(
      (straddle_cost / math.sqrt(max(chosen_dte, 1))) * 1.25, 1
  )

  def doi_t1(k: int, ot: str) -> int:
    cur_oi = int((ce_map if ot == "CE" else pe_map).get(k, {}).get("oi", 0))
    if (k, ot) in snap_t1:
      return cur_oi - snap_t1[(k, ot)]["oi"]
    return 0

  def doi_t2(k: int, ot: str) -> int:
    if (k, ot) in snap_t1 and (k, ot) in snap_t2:
      return snap_t1[(k, ot)]["oi"] - snap_t2[(k, ot)]["oi"]
    return 0

  def pct_oi_t1(k: int, ot: str) -> float:
    if (k, ot) in snap_t1 and snap_t1[(k, ot)]["oi"] > 0:
      return (doi_t1(k, ot) / snap_t1[(k, ot)]["oi"]) * 100.0
    return 0.0

  def pct_oi_t2(k: int, ot: str) -> float:
    if (k, ot) in snap_t2 and snap_t2[(k, ot)]["oi"] > 0:
      return (doi_t2(k, ot) / snap_t2[(k, ot)]["oi"]) * 100.0
    return 0.0

  def dvol_t1(k: int, ot: str) -> int:
    cur_v = int((ce_map if ot == "CE" else pe_map).get(k, {}).get("volume", 0))
    if (k, ot) in snap_t1:
      return max(cur_v - snap_t1[(k, ot)]["vol"], 0)
    return 0

  def dltp_t1(k: int, ot: str) -> float:
    cur_l = float((ce_map if ot == "CE" else pe_map).get(k, {}).get("ltp", 0.0))
    if (k, ot) in snap_t1:
      return round(cur_l - snap_t1[(k, ot)]["ltp"], 2)
    return 0.0

  # ==========================================================================
  # G. EVALUATE ALL 9 CHECKS WITH LAYER-2 CORRECTIONS
  # ==========================================================================
  near_strikes = [s for s in strikes if abs(s - atm) <= 3 * NIFTY_STRIKE_STEP]
  ce_up_strike = min(atm + NIFTY_STRIKE_STEP, strikes[-1])
  pe_dn_strike = max(atm - NIFTY_STRIKE_STEP, strikes[0])
  bull_strike, bear_strike = pe_dn_strike, ce_up_strike

  # --- CHECK #1: OI Wall Gap Ratio (ATM ± 3) + Mandatory Price Confirmation ---
  sum_ce_oi = sum(int(ce_map[s]["oi"]) for s in near_strikes if s in ce_map)
  sum_pe_oi = sum(int(pe_map[s]["oi"]) for s in near_strikes if s in pe_map)
  pe_ce_ratio = round(sum_pe_oi / max(sum_ce_oi, 1), 2)
  ce_pe_ratio = round(sum_ce_oi / max(sum_pe_oi, 1), 2)

  bull_price_conf_c1 = (eff_fut_for_greeks >= put_wall) and (
      fut_price_chg_3m > 0
  )
  bear_price_conf_c1 = (eff_fut_for_greeks <= call_wall) and (
      fut_price_chg_3m < 0
  )
  b_c1 = (pe_ce_ratio >= th["wall_gap_ratio_bull"]) and bull_price_conf_c1
  s_c1 = (ce_pe_ratio >= th["wall_gap_ratio_bear"]) and bear_price_conf_c1

  # --- CHECK #2: Opposing Wall Cut (Fixed Strike, 2x 3m Confirmations, Reduction vs Migration) ---
  fixed_ce_wall_k = (
      atm
      if int(ce_map.get(atm, {}).get("oi", 0))
      >= int(ce_map.get(ce_up_strike, {}).get("oi", 0))
      else ce_up_strike
  )
  fixed_pe_wall_k = (
      atm
      if int(pe_map.get(atm, {}).get("oi", 0))
      >= int(pe_map.get(pe_dn_strike, {}).get("oi", 0))
      else pe_dn_strike
  )

  ce_cut_t1, ce_cut_t2 = pct_oi_t1(fixed_ce_wall_k, "CE"), pct_oi_t2(
      fixed_ce_wall_k, "CE"
  )
  pe_cut_t1, pe_cut_t2 = pct_oi_t1(fixed_pe_wall_k, "PE"), pct_oi_t2(
      fixed_pe_wall_k, "PE"
  )

  ce_2bar_cut = (
      has_2bar_sync
      and (ce_cut_t1 <= th["opposing_cut_pct_3m"])
      and (ce_cut_t2 < 0.0)
  )
  pe_2bar_cut = (
      has_2bar_sync
      and (pe_cut_t1 <= th["opposing_cut_pct_3m"])
      and (pe_cut_t2 < 0.0)
  )

  next_higher_ce = min(fixed_ce_wall_k + NIFTY_STRIKE_STEP, strikes[-1])
  next_lower_pe = max(fixed_pe_wall_k - NIFTY_STRIKE_STEP, strikes[0])
  ce_cut_type = (
      "WALL_MIGRATION_UP"
      if (ce_2bar_cut and doi_t1(next_higher_ce, "CE") > 0)
      else ("NET_WALL_REDUCTION" if ce_2bar_cut else "NO_2BAR_CUT")
  )
  pe_cut_type = (
      "WALL_MIGRATION_DN"
      if (pe_2bar_cut and doi_t1(next_lower_pe, "PE") > 0)
      else ("NET_WALL_REDUCTION" if pe_2bar_cut else "NO_2BAR_CUT")
  )
  b_c2 = ce_2bar_cut
  s_c2 = pe_2bar_cut

  # --- CHECK #3: Supporting OI Speed (3m dOI / Time-of-Day Signed Baseline + Price/LTP Corroboration) ---
  cur_pe_3m_add = max(doi_t1(atm, "PE"), doi_t1(pe_dn_strike, "PE"))
  cur_ce_3m_add = max(doi_t1(atm, "CE"), doi_t1(ce_up_strike, "CE"))
  pe_add_strike = (
      atm
      if doi_t1(atm, "PE") >= doi_t1(pe_dn_strike, "PE")
      else pe_dn_strike
  )
  ce_add_strike = (
      atm
      if doi_t1(atm, "CE") >= doi_t1(ce_up_strike, "CE")
      else ce_up_strike
  )

  cross_sec_pe_pos = [
      doi_t1(s, "PE") for s in near_strikes if doi_t1(s, "PE") > 0
  ]
  cross_sec_ce_pos = [
      doi_t1(s, "CE") for s in near_strikes if doi_t1(s, "CE") > 0
  ]
  pe_signed_base = (
      float(np.mean(hist_pos_pe_doi))
      if len(hist_pos_pe_doi) >= 3
      else max(
          float(np.mean(cross_sec_pe_pos)) if cross_sec_pe_pos else 1.0,
          float(NIFTY_LOT_SIZE * 20),
      )
  )
  ce_signed_base = (
      float(np.mean(hist_pos_ce_doi))
      if len(hist_pos_ce_doi) >= 3
      else max(
          float(np.mean(cross_sec_ce_pos)) if cross_sec_ce_pos else 1.0,
          float(NIFTY_LOT_SIZE * 20),
      )
  )

  pe_speed_mult = (
      round(cur_pe_3m_add / max(pe_signed_base, 1.0), 2)
      if cur_pe_3m_add > 0
      else 0.0
  )
  ce_speed_mult = (
      round(cur_ce_3m_add / max(ce_signed_base, 1.0), 2)
      if cur_ce_3m_add > 0
      else 0.0
  )

  pe_writing_corroborated = (cur_pe_3m_add > 0) and (
      dltp_t1(pe_add_strike, "PE") <= 0.0 and fut_price_chg_3m >= 0.0
  )
  ce_writing_corroborated = (cur_ce_3m_add > 0) and (
      dltp_t1(ce_add_strike, "CE") <= 0.0 and fut_price_chg_3m <= 0.0
  )
  b_c3 = (pe_speed_mult >= th["supporting_oi_speed"]) and pe_writing_corroborated
  s_c3 = (ce_speed_mult >= th["supporting_oi_speed"]) and ce_writing_corroborated

  # --- CHECK #4: Unit-Consistent 3m Churn Ratio (3m Vol Lots / |3m dOI Lots|) + Min dOI Floor ---
  pe_dvol_lots = to_contract_lots(
      dvol_t1(pe_add_strike, "PE"), NIFTY_LOT_SIZE
  )
  pe_doi_lots = abs(to_contract_lots(cur_pe_3m_add, NIFTY_LOT_SIZE))
  ce_dvol_lots = to_contract_lots(
      dvol_t1(ce_add_strike, "CE"), NIFTY_LOT_SIZE
  )
  ce_doi_lots = abs(to_contract_lots(cur_ce_3m_add, NIFTY_LOT_SIZE))

  if pe_doi_lots < th["min_doi_lots_for_churn"]:
    bull_churn = 99.9
    b_c4 = False
  else:
    bull_churn = round(pe_dvol_lots / pe_doi_lots, 2)
    b_c4 = bull_churn <= th["max_churn_ratio"]

  if ce_doi_lots < th["min_doi_lots_for_churn"]:
    bear_churn = 99.9
    s_c4 = False
  else:
    bear_churn = round(ce_dvol_lots / ce_doi_lots, 2)
    s_c4 = bear_churn <= th["max_churn_ratio"]

  # --- CHECK #5: Futures vs Futures VWAP + Index-Weighted Top-5 Breadth ---
  b_c5 = (
      futures_fresh
      and (fut_close > fut_vwap)
      and (norm_bull_breadth >= th["min_weighted_breadth"])
      and (l2_trigger_side in (None, "BULL"))
  )
  s_c5 = (
      futures_fresh
      and (fut_close < fut_vwap)
      and (norm_bear_breadth >= th["min_weighted_breadth"])
      and (l2_trigger_side in (None, "BEAR"))
  )

  # --- CHECK #6: Synchronized Futures Quadrant + Separate Option IV & India VIX Audit ---
  if fut_price_chg_3m > 0 and fut_oi_chg_3m > 0:
    fut_quad = "LONG_BUILDUP"
  elif fut_price_chg_3m > 0 and fut_oi_chg_3m < 0:
    fut_quad = "SHORT_COVERING"
  elif fut_price_chg_3m < 0 and fut_oi_chg_3m > 0:
    fut_quad = "SHORT_BUILDUP"
  elif fut_price_chg_3m < 0 and fut_oi_chg_3m < 0:
    fut_quad = "LONG_UNWINDING"
  else:
    fut_quad = "FLAT_UNCONFIRMED"

  atm_ce_iv_chg = (
      round(
          ce_map[atm]["greeks"]["iv"] - snap_t1[(atm, "CE")]["iv"], 2
      )
      if (atm, "CE") in snap_t1
      else 0.0
  )
  atm_pe_iv_chg = (
      round(
          pe_map[atm]["greeks"]["iv"] - snap_t1[(atm, "PE")]["iv"], 2
      )
      if (atm, "PE") in snap_t1
      else 0.0
  )

  b_c6 = (
      futures_fresh
      and (fut_quad in ("LONG_BUILDUP", "SHORT_COVERING"))
      and (atm_ce_iv_chg >= -0.50)
      and (vix_chp <= 4.0)
  )
  s_c6 = (
      futures_fresh
      and (fut_quad in ("SHORT_BUILDUP", "LONG_UNWINDING"))
      and (atm_pe_iv_chg >= -0.25)
      and (vix_chp >= -1.0)
  )

  # --- CHECK #7: ATM 3-Minute Strike Flip (Fixed Strike K, Price Cross, Volume & 2-Bar Persistence) ---
  prev_spot_t1 = float(header_t1[1]) if header_t1 is not None else spot
  prev_spot_t2 = float(header_t2[1]) if header_t2 is not None else prev_spot_t1
  fixed_pivot_k = min(strikes, key=lambda s: abs(s - prev_spot_t1))

  pe_oi_k_now = int(pe_map.get(fixed_pivot_k, {}).get("oi", 0))
  ce_oi_k_now = int(ce_map.get(fixed_pivot_k, {}).get("oi", 0))
  pe_oi_k_t1 = snap_t1.get((fixed_pivot_k, "PE"), {"oi": pe_oi_k_now})["oi"]
  ce_oi_k_t1 = snap_t1.get((fixed_pivot_k, "CE"), {"oi": ce_oi_k_now})["oi"]
  pe_oi_k_t2 = snap_t2.get((fixed_pivot_k, "PE"), {"oi": pe_oi_k_t1})["oi"]
  ce_oi_k_t2 = snap_t2.get((fixed_pivot_k, "CE"), {"oi": ce_oi_k_t1})["oi"]

  bull_oi_persist = (
      has_2bar_sync
      and (pe_oi_k_now > ce_oi_k_now)
      and (doi_t1(fixed_pivot_k, "PE") > doi_t1(fixed_pivot_k, "CE"))
      and ((pe_oi_k_t1 - ce_oi_k_t1) > (pe_oi_k_t2 - ce_oi_k_t2))
  )
  bear_oi_persist = (
      has_2bar_sync
      and (ce_oi_k_now > pe_oi_k_now)
      and (doi_t1(fixed_pivot_k, "CE") > doi_t1(fixed_pivot_k, "PE"))
      and ((ce_oi_k_t1 - pe_oi_k_t1) > (ce_oi_k_t2 - pe_oi_k_t2))
  )

  bull_price_persist = (spot > fixed_pivot_k) and (
      prev_spot_t2 <= fixed_pivot_k or prev_spot_t1 <= fixed_pivot_k or spot > prev_spot_t1
  )
  bear_price_persist = (spot < fixed_pivot_k) and (
      prev_spot_t2 >= fixed_pivot_k or prev_spot_t1 >= fixed_pivot_k or spot < prev_spot_t1
  )

  avg_3m_vol = max(
      sum(dvol_t1(s, "CE") + dvol_t1(s, "PE") for s in near_strikes)
      / max(len(near_strikes) * 2, 1),
      1.0,
  )
  pe_vol_mult_k = round(dvol_t1(fixed_pivot_k, "PE") / avg_3m_vol, 2)
  ce_vol_mult_k = round(dvol_t1(fixed_pivot_k, "CE") / avg_3m_vol, 2)

  b_c7 = (
      bull_oi_persist
      and bull_price_persist
      and (pe_vol_mult_k >= th["atm_flip_vol_mult"])
  )
  s_c7 = (
      bear_oi_persist
      and bear_price_persist
      and (ce_vol_mult_k >= th["atm_flip_vol_mult"])
  )

  if doi_t1(atm, "PE") > 0 and dltp_t1(atm, "PE") <= 0:
    atm_candidate_label = f"PE_WRITING_CANDIDATE (+{doi_t1(atm, 'PE'):,})"
  elif doi_t1(atm, "CE") > 0 and dltp_t1(atm, "CE") <= 0:
    atm_candidate_label = f"CE_WRITING_CANDIDATE (+{doi_t1(atm, 'CE'):,})"
  elif doi_t1(atm, "CE") > 0 and dltp_t1(atm, "CE") > 0:
    atm_candidate_label = f"CE_BUYING_CANDIDATE (+{doi_t1(atm, 'CE'):,})"
  elif doi_t1(atm, "PE") > 0 and dltp_t1(atm, "PE") > 0:
    atm_candidate_label = f"PE_BUYING_CANDIDATE (+{doi_t1(atm, 'PE'):,})"
  elif doi_t1(atm, "CE") < 0:
    atm_candidate_label = f"CE_UNWINDING_CANDIDATE ({doi_t1(atm, 'CE'):,})"
  elif doi_t1(atm, "PE") < 0:
    atm_candidate_label = f"PE_UNWINDING_CANDIDATE ({doi_t1(atm, 'PE'):,})"
  else:
    atm_candidate_label = "NEUTRAL_OI_ACTIVITY"

  wall_owner_now = (
      f"{fixed_pivot_k} PE_DOM"
      if pe_oi_k_now >= ce_oi_k_now
      else f"{fixed_pivot_k} CE_DOM"
  )
  atm_burst_summary = f"PE_V:{pe_vol_mult_k}x | CE_V:{ce_vol_mult_k}x"

  # --- CHECK #8: Scenario Gamma Flip & VEX/CEX Context ---
  b_c8 = (eff_fut_for_greeks >= gamma_flip and net_gex_cr >= 0) or (
      eff_fut_for_greeks >= call_wall and doi_t1(call_wall, "CE") < 0
  )
  s_c8 = (eff_fut_for_greeks < gamma_flip and net_gex_cr < 0) or (
      eff_fut_for_greeks <= put_wall and doi_t1(put_wall, "PE") < 0
  )

  # --- CHECK #9: Fixed-Strike Migration + Clock Filter ---
  bull_fixed_migration = has_2bar_sync and (
      (doi_t1(atm, "CE") < 0 and doi_t1(ce_up_strike, "CE") > 0)
      or (doi_t1(pe_dn_strike, "PE") < 0 and doi_t1(atm, "PE") > 0)
  )
  bear_fixed_migration = has_2bar_sync and (
      (doi_t1(atm, "PE") < 0 and doi_t1(pe_dn_strike, "PE") < 0)
      or (doi_t1(ce_up_strike, "CE") < 0 and doi_t1(atm, "CE") > 0)
  )

  cur_t = now_dt.time()
  if cur_t < dtime(9, 30):
    clock_regime = "09:15-09:30_OPEN_DISCOVERY_FILTER"
  elif cur_t <= dtime(11, 30):
    clock_regime = "09:30-11:30_MORNING_WINDOW_FILTER"
  elif cur_t <= dtime(14, 30):
    clock_regime = "11:30-14:30_MIDDAY_WINDOW_FILTER"
  elif cur_t < dtime(15, 15):
    clock_regime = "14:30-15:15_LATE_WINDOW_FILTER"
  else:
    clock_regime = "15:15-15:30_CLOSING_FREEZE_VETO"

  not_in_freeze = dtime(9, 20) <= cur_t < dtime(15, 15)
  b_c9 = not_in_freeze and open_gap_bull and bull_fixed_migration
  s_c9 = not_in_freeze and open_gap_bear and bear_fixed_migration

  migration_str = (
      "BULL_WALL_MIGRATION_UP"
      if bull_fixed_migration
      else (
          "BEAR_WALL_MIGRATION_DN"
          if bear_fixed_migration
          else "NO_CONFIRMED_MIGRATION"
      )
  )

  # ==========================================================================
  # H. STRICT 3-STATE FINAL VERDICT: [ 🟢 BULL | 🔴 BEAR | ⚪ NO TRADE ]
  # ==========================================================================
  bull_list = [b_c1, b_c2, b_c3, b_c4, b_c5, b_c6, b_c7, b_c8, b_c9]
  bear_list = [s_c1, s_c2, s_c3, s_c4, s_c5, s_c6, s_c7, s_c8, s_c9]
  total_checks = 9
  bull_score = sum(1 for x in bull_list if x)
  bear_score = sum(1 for x in bear_list if x)

  bull_pillar_oi = ((b_c1 or b_c3) and (b_c2 or b_c7 or b_c9)) or (b_c1 and b_c3)
  bear_pillar_oi = ((s_c1 or s_c3) and (s_c2 or s_c7 or s_c9)) or (s_c1 and s_c3)
  bull_pillar_vol = b_c4
  bear_pillar_vol = s_c4
  bull_pillar_fut = b_c5 and b_c6
  bear_pillar_fut = s_c5 and s_c6
  bull_pillar_gex = b_c8
  bear_pillar_gex = s_c8

  bull_pillars = sum(
      [bull_pillar_oi, bull_pillar_vol, bull_pillar_fut, bull_pillar_gex]
  )
  bear_pillars = sum(
      [bear_pillar_oi, bear_pillar_vol, bear_pillar_fut, bear_pillar_gex]
  )

  no_trade_reasons = []
  freshness_ok = (
      spot > 0
      and vix_ltp > 0
      and futures_fresh
      and quotes_fresh
      and chosen_dte >= 1
  )
  if not freshness_ok:
    no_trade_reasons.append("STALE_OR_MISSING_CRITICAL_DATA")
  if not not_in_freeze:
    no_trade_reasons.append("OUTSIDE_TRADING_HOURS_OR_1515_FREEZE")

  bull_contradiction = (fut_close < fut_vwap) or (fut_price_chg_3m < 0)
  bear_contradiction = (fut_close > fut_vwap) or (fut_price_chg_3m > 0)

  # Strict 3-State Decision Output: ONLY "BULL", "BEAR", or "NO TRADE"
  final_verdict = "NO TRADE"
  active_side = "NONE"
  is_actionable = False

  if freshness_ok and not_in_freeze:
    if (
        bull_pillar_oi
        and bull_pillar_fut
        and not bull_contradiction
        and bear_pillars <= 1
    ):
      final_verdict = "BULL"
      active_side = "BULL"
      is_actionable = True
      grade = "STRONG " if (bull_pillars == 4 and has_2bar_sync) else ""
      signal_banner = f"🟢 BULL — {grade}BUY {ACTIVE_INDEX} CE (Pillars {bull_pillars}/4 | Checks {bull_score}/9)"
    elif (
        bear_pillar_oi
        and bear_pillar_fut
        and not bear_contradiction
        and bull_pillars <= 1
    ):
      final_verdict = "BEAR"
      active_side = "BEAR"
      is_actionable = True
      grade = "STRONG " if (bear_pillars == 4 and has_2bar_sync) else ""
      signal_banner = f"🔴 BEAR — {grade}BUY {ACTIVE_INDEX} PE (Pillars {bear_pillars}/4 | Checks {bear_score}/9)"

  if final_verdict == "NO TRADE":
    if (b_c1 or b_c3) and bull_contradiction:
      no_trade_reasons.append("BULL_VETOED_BY_FUTURES")
    if (s_c1 or s_c3) and bear_contradiction:
      no_trade_reasons.append("BEAR_VETOED_BY_FUTURES")
    if not has_2bar_sync:
      no_trade_reasons.append("WARMUP_WAITING_2_BARS")
    if not no_trade_reasons:
      no_trade_reasons.append(
          f"INSUFFICIENT_ALIGNMENT (Bull {bull_pillars}/4 [{bull_score}/9] | Bear {bear_pillars}/4 [{bear_score}/9])"
      )
    signal_banner = f"⚪ NO TRADE — {' | '.join(no_trade_reasons)}"

  # Symmetric Contract Selection
  if active_side == "BULL":
    target_strike = bull_strike
    leg = ce_map.get(target_strike, ce_map[atm])
    opt_type_rec = "CE"
  elif active_side == "BEAR":
    target_strike = bear_strike
    leg = pe_map.get(target_strike, pe_map[atm])
    opt_type_rec = "PE"
  else:
    target_strike = atm
    leg = ce_map.get(atm, {})
    opt_type_rec = "NONE"

  g = leg.get(
      "greeks",
      {"delta": 0.5, "gamma": 0.001, "theta": -15.0, "vega": 5.0, "iv": 14.0},
  )
  entry_ask = (
      float(leg.get("ask", 0) or leg.get("ltp", 100.0))
      if opt_type_rec != "NONE"
      else 0.0
  )
  bid_p = (
      float(leg.get("bid", 0) or entry_ask - 0.35)
      if opt_type_rec != "NONE"
      else 0.0
  )
  spread = round(max(entry_ask - bid_p, 0.25), 2) if opt_type_rec != "NONE" else 0.0

  if is_actionable and spread > MAX_ALLOWED_SPREAD:
    final_verdict = "NO TRADE"
    active_side = "NONE"
    is_actionable = False
    opt_type_rec = "NONE"
    no_trade_reasons.append(f"WIDE_BID_ASK_SPREAD_{spread}")
    signal_banner = f"⚪ NO TRADE — Wide Option Spread (₹{spread} > ₹{MAX_ALLOWED_SPREAD})"

  stop_dist = (
      abs(spot - custom_stop_spot) if custom_stop_spot else DEFAULT_STOP_PTS
  )
  gamma_cushion = 0.5 * float(g.get("gamma", 0.0)) * (stop_dist**2)
  loss_unit = max(
      (abs(float(g.get("delta", 0.5))) * stop_dist)
      - gamma_cushion
      + (abs(float(g.get("theta", 0.0))) * 0.25)
      + spread,
      2.0,
  )
  opt_sl = round(max(entry_ask - loss_unit, 1.0), 2) if opt_type_rec != "NONE" else 0.0
  loss_lot_rs = loss_unit * NIFTY_LOT_SIZE
  risk_budget = float(os.environ.get("RISK_PER_TRADE_RS", "7500.0"))
  lots = (
      max(math.floor(risk_budget / max(loss_lot_rs, 1.0)), 1)
      if opt_type_rec != "NONE"
      else 0
  )
  est_risk_rs = round(lots * loss_lot_rs, 0) if opt_type_rec != "NONE" else 0.0
  rec_contract_str = (
      leg.get("symbol", "NONE") if opt_type_rec != "NONE" else "NONE (NO TRADE)"
  )

  trade_plan = {
      "final_verdict": final_verdict,
      "decision_outcome": final_verdict,
      "active_side": active_side,
      "contract": rec_contract_str,
      "strike": target_strike,
      "type": opt_type_rec,
      "expiry": chosen_exp["date"],
      "dte": chosen_dte,
      "entry_ask": entry_ask,
      "spread_rs": spread,
      "option_stop_price": opt_sl,
      "spot_stop_pts": round(stop_dist, 1),
      "lots": lots,
      "qty": lots * NIFTY_LOT_SIZE,
      "est_risk_rs": est_risk_rs,
      "velocity_stop_mins": 45 if active_side == "BULL" else 30,
  }

  # I. Save Layer-2 Corrected 9-Check Scorecard
  scorecard_rows = [
      {
          "Check": "1. Wall Gap Ratio + Price Conf",
          "Bull_Live": (
              f"PE/CE={pe_ce_ratio} (>={th['wall_gap_ratio_bull']}) &"
              f" PxConf:{bull_price_conf_c1}"
          ),
          "Bull_Status": "PASS" if b_c1 else "FAIL",
          "Bear_Live": (
              f"CE/PE={ce_pe_ratio} (>={th['wall_gap_ratio_bear']}) &"
              f" PxConf:{bear_price_conf_c1}"
          ),
          "Bear_Status": "PASS" if s_c1 else "FAIL",
      },
      {
          "Check": "2. Opposing Cut (2x 3m Fixed K)",
          "Bull_Live": (
              f"{fixed_ce_wall_k}CE: {ce_cut_t1:.1f}%/{ce_cut_t2:.1f}%"
              f" ({ce_cut_type})"
          ),
          "Bull_Status": "PASS" if b_c2 else "FAIL",
          "Bear_Live": (
              f"{fixed_pe_wall_k}PE: {pe_cut_t1:.1f}%/{pe_cut_t2:.1f}%"
              f" ({pe_cut_type})"
          ),
          "Bear_Status": "PASS" if s_c2 else "FAIL",
      },
      {
          "Check": "3. Supporting OI Speed (ToD Base)",
          "Bull_Live": (
              f"{pe_speed_mult}x (>={th['supporting_oi_speed']}x) |"
              f" Corrob:{pe_writing_corroborated}"
          ),
          "Bull_Status": "PASS" if b_c3 else "FAIL",
          "Bear_Live": (
              f"{ce_speed_mult}x (>={th['supporting_oi_speed']}x) |"
              f" Corrob:{ce_writing_corroborated}"
          ),
          "Bear_Status": "PASS" if s_c3 else "FAIL",
      },
      {
          "Check": "4. 3m Churn (Lots Vol / |dOI|)",
          "Bull_Live": (
              f"{bull_churn} (<={th['max_churn_ratio']},"
              f" dOI={pe_doi_lots:.0f}L)"
          ),
          "Bull_Status": "PASS" if b_c4 else "FAIL",
          "Bear_Live": (
              f"{bear_churn} (<={th['max_churn_ratio']},"
              f" dOI={ce_doi_lots:.0f}L)"
          ),
          "Bear_Status": "PASS" if s_c4 else "FAIL",
      },
      {
          "Check": "5. Fut vs FutVWAP + Wt Breadth*",
          "Bull_Live": f"Fut>{fut_vwap} & WtBull={norm_bull_breadth:.0%}",
          "Bull_Status": "PASS" if b_c5 else "FAIL",
          "Bear_Live": f"Fut<{fut_vwap} & WtBear={norm_bear_breadth:.0%}",
          "Bear_Status": "PASS" if s_c5 else "FAIL",
      },
      {
          "Check": "6. Fut Quad (3m) + OptIV / VIX",
          "Bull_Live": (
              f"{fut_quad} | CE_dIV:{atm_ce_iv_chg:+.2f} | VIX:{vix_chp:+.1f}%"
          ),
          "Bull_Status": "PASS" if b_c6 else "FAIL",
          "Bear_Live": (
              f"{fut_quad} | PE_dIV:{atm_pe_iv_chg:+.2f} | VIX:{vix_chp:+.1f}%"
          ),
          "Bear_Status": "PASS" if s_c6 else "FAIL",
      },
      {
          "Check": f"7. Fixed-K ({fixed_pivot_k}) 2x3m Flip",
          "Bull_Live": (
              f"Persist:{bull_oi_persist} | PxCross:{bull_price_persist} |"
              f" V:{pe_vol_mult_k}x"
          ),
          "Bull_Status": "PASS" if b_c7 else "FAIL",
          "Bear_Live": (
              f"Persist:{bear_oi_persist} | PxCross:{bear_price_persist} |"
              f" V:{ce_vol_mult_k}x"
          ),
          "Bear_Status": "PASS" if s_c7 else "FAIL",
      },
      {
          "Check": f"8. Scenario γ-Flip ({gamma_flip})",
          "Bull_Live": f"GEX:{net_gex_cr:+.0f}Cr/1% | VEX:{net_vex_cr:+.1f}Cr",
          "Bull_Status": "PASS" if b_c8 else "FAIL",
          "Bear_Live": f"GEX:{net_gex_cr:+.0f}Cr/1% | CEX:{net_cex_cr:+.1f}Cr",
          "Bear_Status": "PASS" if s_c8 else "FAIL",
      },
      {
          "Check": "9. Fixed Migration + Clock Filter",
          "Bull_Live": f"{migration_str} | {clock_regime[:11]}",
          "Bull_Status": "PASS" if b_c9 else "FAIL",
          "Bear_Live": f"{migration_str} | {clock_regime[:11]}",
          "Bear_Status": "PASS" if s_c9 else "FAIL",
      },
  ]
  pd.DataFrame(scorecard_rows).to_csv(SCORECARD_CSV_PATH, index=False)

  bull_flags_str = " ".join(
      [f"C{i+1}:{'Y' if v else 'N'}" for i, v in enumerate(bull_list)]
  )
  bear_flags_str = " ".join(
      [f"C{i+1}:{'Y' if v else 'N'}" for i, v in enumerate(bear_list)]
  )

  # K. Append to Original 27-Col CSV & Layer-2 Audit CSV
  with open(SIGNALS_CSV_PATH, "a", newline="") as f:
    csv.writer(f).writerow([
        ist_str,
        data_mode,
        signal_banner,
        f"{bull_score}/{total_checks}",
        f"{bear_score}/{total_checks}",
        bull_flags_str,
        bear_flags_str,
        spot,
        fut_close,
        fut_vwap,
        fut_quad,
        vix_ltp,
        vix_chp,
        round(net_gex_cr, 2),
        gex_regime,
        put_wall,
        call_wall,
        wall_owner_now,
        atm_candidate_label,
        atm_burst_summary,
        bull_cnt,
        bear_cnt,
        rec_contract_str,
        entry_ask,
        opt_sl,
        lots,
        est_risk_rs,
    ])
    f.flush()
    os.fsync(f.fileno())

  with open(AUDIT_CSV_PATH, "a", newline="") as af_csv:
    csv.writer(af_csv).writerow([
        ist_str,
        final_verdict,
        signal_banner,
        active_side,
        bull_score,
        bear_score,
        f"{bull_pillars}/4",
        f"{bear_pillars}/4",
        "|".join(no_trade_reasons) or "NONE",
        freshness_ok,
        spot,
        fut_close,
        fut_vwap,
        norm_bull_breadth,
        norm_bear_breadth,
        f"CE:{ce_cut_type}|PE:{pe_cut_type}",
        atm_candidate_label,
        round(net_gex_cr, 2),
        round(net_vex_cr, 2),
        round(net_cex_cr, 2),
        gamma_flip,
        rec_contract_str,
    ])
    af_csv.flush()
    os.fsync(af_csv.fileno())
  os.sync()

  # G.7: Separate auction research poll; fail closed without modifying the live gate.
  try:
    auction_history = fyers.history(
        data={
            "symbol": NIFTY_SPOT_SYM,
            "resolution": "5",
            "date_format": "1",
            "range_from": (today_ist - timedelta(days=12)).isoformat(),
            "range_to": today_ist.isoformat(),
            "cont_flag": "1",
        }
    )
    if auction_history.get("s") not in (None, "ok") or not auction_history.get(
        "candles"
    ):
      raise ValueError("FYERS history returned no valid candles")
    auction_state = evaluate(auction_history["candles"], now=now_dt)
  except Exception as auction_error:
    auction_state = {
        "status": "INSUFFICIENT_DATA",
        "decision": "NO_TRADE",
        "reason": str(auction_error),
        "events": [],
        "previous_tpo": None,
    }

  LATEST_STATE = {
      "status": "OK",
      "error": None,
      "final_verdict": final_verdict,
      "banner": signal_banner,
      "decision_outcome": final_verdict,
      "scorecard": scorecard_rows,
      "trade_plan": trade_plan,
      "auction_research": auction_state,
      "layer2_audit": {
          "bull_pillars": f"{bull_pillars}/4",
          "bear_pillars": f"{bear_pillars}/4",
          "no_trade_reasons": no_trade_reasons,
          "freshness_ok": freshness_ok,
          "gex_units": f"{net_gex_cr:+.2f} ₹Cr / 1% Spot (OI-Signed Scenario)",
          "vex_units": f"{net_vex_cr:+.2f} ₹Cr / 1% IV (OI-Signed Scenario)",
          "cex_units": f"{net_cex_cr:+.2f} ₹Cr / Day (OI-Signed Scenario)",
      },
      "meta": {
          "ist_str": ist_str,
          "fut_sym": fut_sym,
          "data_mode": data_mode,
          "spot": spot,
          "gamma_flip": gamma_flip,
          "fut_close": fut_close,
          "fut_vwap": fut_vwap,
          "vix": f"{vix_ltp} ({vix_chp}%)",
          "straddle_idr": implied_daily_range,
          "gex": f"{net_gex_cr:+.2f} Cr/1% ({gex_regime})",
          "vex_cex": f"VEX {net_vex_cr:+.1f} Cr/IV | CEX {net_cex_cr:+.1f} Cr/d",
          "put_wall": put_wall,
          "call_wall": call_wall,
          "atm_wall": wall_owner_now,
          "seller_lead": atm_candidate_label,
          "migration": migration_str,
          "clock_regime": clock_regime,
          "bull_breadth": f"{bull_cnt}/5 (Wt {norm_bull_breadth:.0%})",
          "bear_breadth": f"{bear_cnt}/5 (Wt {norm_bear_breadth:.0%})",
          "bull_score": f"{bull_score}/9 (Pillars {bull_pillars}/4)",
          "bear_score": f"{bear_score}/9 (Pillars {bear_pillars}/4)",
          "contract": rec_contract_str,
          "dte": chosen_dte,
          "ask": entry_ask,
          "spread": spread,
          "sl": opt_sl,
          "lots": lots,
          "qty": trade_plan["qty"],
          "risk": f"{est_risk_rs:.0f}",
      },
  }

  if is_actionable:
    send_telegram_alert(LATEST_STATE)

  # Visually distinct terminal output for BULL vs BEAR vs NO TRADE
  if final_verdict == "BULL":
    print("\n" + "🟢" * 36)
    print(f"🚀 FINAL VERDICT: [ BULL ] | {signal_banner}")
    print(
        f"🎯 BUY CE: {rec_contract_str} @ Ask ₹{entry_ask} | SL: ₹{opt_sl} |"
        f" Lots: {lots} ({trade_plan['qty']}u) | Fut ({fut_sym}): {fut_close}"
    )
    print("🟢" * 36 + "\n")
  elif final_verdict == "BEAR":
    print("\n" + "🔴" * 36)
    print(f"🔻 FINAL VERDICT: [ BEAR ] | {signal_banner}")
    print(
        f"🎯 BUY PE: {rec_contract_str} @ Ask ₹{entry_ask} | SL: ₹{opt_sl} |"
        f" Lots: {lots} ({trade_plan['qty']}u) | Fut ({fut_sym}): {fut_close}"
    )
    print("🔴" * 36 + "\n")
  else:
    print(
        f"⚪ [{ist_str}] [ NO TRADE ] | {signal_banner} | Fut ({fut_sym}):"
        f" {fut_close} vs VWAP: {fut_vwap} | {atm_candidate_label}"
    )

  return LATEST_STATE


def send_telegram_alert(state: Dict[str, Any]):
  bot_token = os.environ.get("TELEGRAM_BOT_TOKEN", "")
  chat_id = os.environ.get("TELEGRAM_CHAT_ID", "")
  if not bot_token or not chat_id or "YOUR_" in bot_token:
    return
  m = state["meta"]
  tp = state["trade_plan"]
  icon = "🟢🚀" if state["final_verdict"] == "BULL" else "🔴🔻"
  msg = (
      f"{icon} *FINAL VERDICT: {state['final_verdict']}*\n"
      f"📋 `{state['banner']}`\n"
      f"🕒 {m['ist_str']} ({m['fut_sym']})\n"
      f"📈 *Fut:* {m['fut_close']} vs *Fut VWAP:* {m['fut_vwap']} | *Spot:*"
      f" {m['spot']}\n"
      f"🧱 *Fixed Pivot:* {m['atm_wall']} ({m['seller_lead']})\n"
      f"🏦 *Weighted Breadth:* Bull {m['bull_breadth']} | Bear"
      f" {m['bear_breadth']}\n"
      f"⚡ *Scenario GEX:* {m['gex']} | *Est Flip:* {m['gamma_flip']}\n\n"
      f"🎯 *Contract:* `{tp['contract']}` ({tp['dte']} DTE)\n"
      f"💰 *Limit Ask:* ₹{tp['entry_ask']} (Spread ₹{tp['spread_rs']}) | *SL:*"
      f" ₹{tp['option_stop_price']}\n"
      f"📦 *Size:* {tp['lots']} Lots ({tp['qty']} units) | *Est Risk:*"
      f" ₹{tp['est_risk_rs']:.0f}"
  )
  try:
    requests.post(
        f"https://api.telegram.org/bot{bot_token}/sendMessage",
        json={"chat_id": chat_id, "text": msg, "parse_mode": "Markdown"},
        timeout=5,
    )
  except Exception:
    pass


# ==============================================================================
# 5. UNIFIED FASTAPI SERVER (/mobile & /webhook/tradingview)
# ==============================================================================
app = FastAPI(title="Layer-2 Corrected 9-Check Institutional Engine")


@app.post("/webhook/tradingview")
async def tradingview_webhook(req: Request):
  payload = await req.json()
  sig = payload.get("signal", "")
  side = "BULL" if "BULL" in sig else ("BEAR" if "BEAR" in sig else None)
  stop_spot = float(payload["stop_spot"]) if "stop_spot" in payload else None
  target_spot = (
      float(payload["target_spot"]) if "target_spot" in payload else None
  )
  return JSONResponse(
      poll_and_evaluate(
          l2_trigger_side=side,
          custom_stop_spot=stop_spot,
          custom_target_spot=target_spot,
      )
  )


@app.get("/mobile", response_class=HTMLResponse)
def mobile_scorecard():
  s = LATEST_STATE
  if s.get("error") and not s["scorecard"]:
    return (
        "<html><body"
        " style='background:#121316;color:#EA4335;font-family:sans-serif;padding:20px;'><h3>⚠️"
        f" NO TRADE / API Error</h3><p>{s['error']}</p></body></html>"
    )
  if not s["scorecard"]:
    return (
        "<html><body"
        " style='background:#121316;color:#fff;font-family:sans-serif;padding:20px;'><h3>Initializing"
        " Layer-2 9-Check scan... refresh in 10s.</h3></body></html>"
    )
  m = s["meta"]
  l2 = s.get("layer2_audit", {})
  verdict = s.get("final_verdict", "NO TRADE")

  # High-contrast visual styling for BULL vs BEAR vs NO TRADE
  if verdict == "BULL":
    badge_bg = "#065F46"
    badge_border = "#10B981"
    badge_text = "🟢 FINAL VERDICT: BULL (BUY CE)"
  elif verdict == "BEAR":
    badge_bg = "#7F1D1D"
    badge_border = "#EF4444"
    badge_text = "🔴 FINAL VERDICT: BEAR (BUY PE)"
  else:
    badge_bg = "#1F2937"
    badge_border = "#6B7280"
    badge_text = "⚪ FINAL VERDICT: NO TRADE"

  rows_html = "".join([
      f"<tr><td style='padding:7px;border-bottom:1px solid #2A2D30;'>{r['Check']}</td>"
      "<td style='padding:7px;border-bottom:1px solid"
      f" #2A2D30;color:{'#34A853' if r['Bull_Status']=='PASS' else '#EA4335'};'><b>{r['Bull_Status']}</b><br/><small"
      f" style='color:#9AA0A6;'>{r['Bull_Live']}</small></td>"
      "<td style='padding:7px;border-bottom:1px solid"
      f" #2A2D30;color:{'#34A853' if r['Bear_Status']=='PASS' else '#EA4335'};'><b>{r['Bear_Status']}</b><br/><small"
      f" style='color:#9AA0A6;'>{r['Bear_Live']}</small></td></tr>"
      for r in s["scorecard"]
  ])
  return f"""
    <html><head><meta name="viewport" content="width=device-width, initial-scale=1"><meta http-equiv="refresh" content="30"></head>
    <body style="font-family:sans-serif;background:#121316;color:#fff;padding:12px;max-width:540px;margin:auto;">
        <div style="background:{badge_bg};border:2px solid {badge_border};padding:12px;border-radius:8px;text-align:center;margin-bottom:10px;">
            <div style="font-size:20px;font-weight:bold;letter-spacing:0.5px;">{badge_text}</div>
            <div style="font-size:12px;color:#E5E7EB;margin-top:4px;">{s['banner']}</div>
        </div>
        <small style="color:#9AA0A6;">{m.get('ist_str')} • Fut: {m.get('fut_sym')} • Sync: {m.get('data_mode')}</small>
        <p style="margin:6px 0;font-size:13px;"><b>Fut:</b> {m.get('fut_close')} vs <b>Fut VWAP:</b> {m.get('fut_vwap')} | <b>Spot:</b> {m.get('spot')} | <b>VIX:</b> {m.get('vix')}</p>
        <p style="margin:6px 0;font-size:12px;color:#A7F3D0;"><b>Weighted Breadth:</b> Bull {m.get('bull_breadth')} | Bear {m.get('bear_breadth')}</p>
        <p style="margin:6px 0;font-size:12px;color:#9AA0A6;"><b>Scenario Units:</b> {l2.get('gex_units')}<br/>{l2.get('vex_units')} | {l2.get('cex_units')}</p>
        <p style="margin:6px 0;font-size:13px;color:#38BDF8;"><b>ATM Activity:</b> {m.get('atm_wall')} • {m.get('seller_lead')}</p>
        <table style="width:100%;border-collapse:collapse;font-size:12px;background:#1E293B;border-radius:6px;">
            <tr style="background:#0F172A;text-align:left;">
                <th style="padding:7px;">Layer-2 Corrected Check</th>
                <th style="padding:7px;">Bull CE ({m.get('bull_score')})</th>
                <th style="padding:7px;">Bear PE ({m.get('bear_score')})</th>
            </tr>
            {rows_html}
        </table>
        <p style="background:#1E293B;padding:10px;border-radius:6px;border-left:4px solid {badge_border};font-size:13px;margin-top:10px;">
            <b>Actionable Contract:</b> {m.get('contract')} ({m.get('dte')} DTE)<br/>
            <b>Limit Ask:</b> ₹{m.get('ask')} (Spread ₹{m.get('spread')}) | <b>2nd-Order SL:</b> ₹{m.get('sl')}<br/>
            <b>Size:</b> {m.get('lots')} Lots ({m.get('qty')} units) | <b>Est. Risk:</b> ₹{m.get('risk')}
        </p>
    </body></html>
    """


# ==============================================================================
# 6. SAFE BOOTSTRAP -> LAUNCH FASTAPI & CLOUDFLARE -> 3-MINUTE LOOP
# ==============================================================================
def get_free_port() -> int:
  with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind(("127.0.0.1", 0))
    return s.getsockname()[1]


init_storage()
print("⏳ Step 1: Running Strict 3-State [ BULL | BEAR | NO TRADE ] scan...")
try:
  poll_and_evaluate()
except Exception as init_err:
  print("⚠️ Initial scan warning (will retry in loop):", init_err)

port = get_free_port()
print(f"📡 Step 2: Starting FastAPI server on port {port}...")
threading.Thread(
    target=lambda: uvicorn.run(
        app, host="0.0.0.0", port=port, log_level="warning"
    ),
    daemon=True,
).start()
time.sleep(1)

cf_bin = (
    "/usr/local/bin/cloudflared"
    if os.path.exists("/usr/local/bin/cloudflared")
    else "cloudflared"
)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
tunnel_p = subprocess.Popen(
    [cf_bin, "tunnel", "--url", f"http://localhost:{port}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

tunnel_url = None
for _ in range(40):
  line = tunnel_p.stdout.readline()
  if not line:
    break
  m_url = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
  if m_url:
    tunnel_url = m_url.group(0)
    break

print("\n" + "=" * 78)
if tunnel_url:
  print(f"📱 LIVE MOBILE DASHBOARD (BULL/BEAR/NO TRADE) : {tunnel_url}/mobile")
  print(f"🔗 TRADINGVIEW WEBHOOK URL                    : {tunnel_url}/webhook/tradingview")
print(f"📂 LAYER-2 AUDIT CSV                          : {AUDIT_CSV_PATH}")
print(f"📂 SIGNALS LOG CSV                            : {SIGNALS_CSV_PATH}")
print("=" * 78 + "\n")

while True:
  time.sleep(180)
  try:
    poll_and_evaluate()
  except Exception as e:
    print(f"[{now_ist().strftime('%H:%M:%S')}] Loop Error: {e}")